In [1]:
import pandas as pd

# Load all 4 CSVs — fill in the correct filenames
event_details = pd.read_csv('../scraped_data/ufc_event_details.csv')
fight_details = pd.read_csv('../scraped_data/ufc_fight_details.csv')
fight_results = pd.read_csv('../scraped_data/ufc_fight_results.csv')
fight_stats = pd.read_csv('../scraped_data/ufc_fight_stats.csv')
fighter_details = pd.read_csv('../scraped_data/ufc_fighter_details.csv')
fighter_tott = pd.read_csv('../scraped_data/ufc_fighter_tott.csv')

# For each dataframe, run these 4 lines and study the output
for name, df in {
    'event_details': event_details,
    'fight_details': fight_details,
    'fight_results': fight_results,
    'fight_stats': fight_stats,
    'fighter_details': fighter_details,
    'fighter_tott': fighter_tott
}.items():
    print(f"\n{'='*40}")
    print(f"DATAFRAME: {name}")
    print(f"{'='*40}")
    print(df.columns.tolist())  
    print(df.shape)
    print(df.info())
    print(df.describe())


DATAFRAME: event_details
['EVENT', 'URL', 'DATE', 'LOCATION']
(763, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 763 entries, 0 to 762
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   EVENT     763 non-null    object
 1   URL       763 non-null    object
 2   DATE      763 non-null    object
 3   LOCATION  763 non-null    object
dtypes: object(4)
memory usage: 24.0+ KB
None
                    EVENT                                                URL  \
count                 763                                                763   
unique                763                                                763   
top     UFC 2: No Way Out  http://ufcstats.com/event-details/a6a9ab5a824e...   
freq                    1                                                  1   

                     DATE                LOCATION  
count                 763                     763  
unique                758                    

Main things to Note from running the above cell:
- DATE is a string, not a datetime
- stats stored as "X of Y" can be separated into landed / attempted
- HEIGHT, WEIGHT, REACH stored as strings
- Missing a lot of values for NICKNAME, STANCE, REFEREE, DETAILS, HEIGHT, and REACH

These issues and more is handled by methods in preprocessing.py in src folder

In [2]:
import sys

if 'src.preprocessing' in sys.modules:
    del sys.modules['src.preprocessing']
if 'src.features' in sys.modules:
    del sys.modules['src.features']

sys.path.append('..')

from src.preprocessing import run_all
from src.features import build_fights

event_details, fight_details, fight_results, fight_stats, fighter_details, fighter_tott = run_all(data_dir='../scraped_data/')

fights = build_fights(fight_results, event_details, fighter_tott)
print(fights.shape)
print(fights.columns.tolist())

(8353, 17)
['WEIGHTCLASS', 'METHOD', 'ROUND', 'TOTAL_TIME', 'FIGHTER_1', 'FIGHTER_2', 'DATE', 'F1_HEIGHT', 'F1_WEIGHT', 'F1_REACH', 'F1_STANCE', 'F2_HEIGHT', 'F2_WEIGHT', 'F2_REACH', 'F2_STANCE', 'F1_AGE', 'F2_AGE']


Note: Did handle blank entries in TD % to be 0, rather than missing, as indicates no takedowns attempted by the fighter. Keep in mind during feature engineering.

In [3]:
from src.features import build_fights

fights = build_fights(fight_results, event_details, fighter_tott)
print(fights.head())
print(fights.dtypes)
print(fights.shape)
print(fights.columns.tolist())

             WEIGHTCLASS                 METHOD  ROUND  TOTAL_TIME  \
0  UFC Heavyweight Title                KO/TKO       3         853   
1            Heavyweight   Decision - Majority       3         900   
2      Super Heavyweight                KO/TKO       2         574   
3            Heavyweight            Submission       1          55   
4           Middleweight  Decision - Unanimous       2         600   

         FIGHTER_1        FIGHTER_2       DATE  F1_HEIGHT  F1_WEIGHT  \
0    Randy Couture  Kevin Randleman 2000-11-17       73.0      205.0   
1    Renato Sobral    Maurice Smith 2000-11-17       73.0      205.0   
2     Josh Barnett        Gan McGee 2000-11-17       75.0      250.0   
3  Andrei Arlovski      Aaron Brink 2000-11-17       75.0      240.0   
4      Mark Hughes   Alex Stiebling 2000-11-17       69.0      205.0   

   F1_REACH F1_STANCE  F2_HEIGHT  F2_WEIGHT  F2_REACH F2_STANCE     F1_AGE  \
0      75.0  Orthodox       70.0      205.0       NaN  Orthodox  37.

In [4]:
from src.features import add_outcome_label
labels = add_outcome_label(fight_results, event_details)
fights = fights.merge(labels, on=['FIGHTER_1', 'FIGHTER_2', 'DATE'], how='left')
print(fights['OUTCOME_LABEL'].value_counts())
print(fights['OUTCOME_LABEL'].isna().sum())

OUTCOME_LABEL
1.0    5188
0.0    3019
2.0      57
Name: count, dtype: int64
89


In [5]:
unmatched = fights[fights['OUTCOME_LABEL'].isna()][['FIGHTER_1', 'FIGHTER_2', 'DATE']]
print(unmatched.head(10))
print(fights[fights['OUTCOME_LABEL'].isna()]['WEIGHTCLASS'].value_counts())
print(fight_results['OUTCOME'].value_counts())

              FIGHTER_1         FIGHTER_2       DATE
18        Bobby Hoffman     Mark Robinson 2001-02-23
75         Benji Radach      Steve Berger 2002-05-10
247      Alessio Sakara     Ron Faircloth 2005-10-07
506        Gray Maynard       Rob Emerson 2007-06-23
813       Karo Parisyan     Dong Hyun Kim 2009-01-31
1268       Brandon Vera      Thiago Silva 2011-01-01
1399          Nik Lentz  Charles Oliveira 2011-06-26
1518  Mackens Semerzier    Robert Peralta 2011-11-12
1773     Chris Clements    Matthew Riddle 2012-07-21
1779     Roland Delorme  Francisco Rivera 2012-07-21
WEIGHTCLASS
Bantamweight                   16
Lightweight                    13
Light Heavyweight              12
Welterweight                   12
Middleweight                   12
Heavyweight                     9
Featherweight                   8
Women's Bantamweight            2
Flyweight                       2
UFC Light Heavyweight Title     1
Catch Weight                    1
UFC Heavyweight Title          

In [6]:
fights = fights[fights['OUTCOME_LABEL'].notna()].reset_index(drop=True)
fights = fights[fights['OUTCOME_LABEL'] != 2].reset_index(drop=True)

In [8]:
print(fights.shape)
print(fights['OUTCOME_LABEL'].value_counts())
print(fights.isna().sum())

(8207, 18)
OUTCOME_LABEL
1.0    5188
0.0    3019
Name: count, dtype: int64
WEIGHTCLASS        0
METHOD             0
ROUND              0
TOTAL_TIME         0
FIGHTER_1          0
FIGHTER_2          0
DATE               0
F1_HEIGHT         24
F1_WEIGHT         17
F1_REACH         233
F1_STANCE         35
F2_HEIGHT         29
F2_WEIGHT         16
F2_REACH         690
F2_STANCE         57
F1_AGE            23
F2_AGE            62
OUTCOME_LABEL      0
dtype: int64
